# Project 99 — Local AI Project Critic
## Project Proposal → Feasibility Analysis → Risk Assessment → Recommendations

**Stack:** LangChain · Ollama · Pydantic · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama pydantic pandas

## Step 1 — Project Proposals to Evaluate

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
import json, pandas as pd

llm = ChatOllama(model="qwen3:8b", temperature=0.2)

proposals = [
    {
        "title": "AI-Powered Customer Support Chatbot",
        "description": "Build a chatbot using RAG to answer customer questions from our knowledge base. "
                       "Handle 80% of tier-1 support tickets automatically.",
        "tech_stack": "LangChain, ChromaDB, Ollama, FastAPI",
        "timeline": "3 months",
        "team_size": 2,
        "budget": "$50K",
        "data_available": "5000 support tickets, 200 FAQ docs, product manuals",
    },
    {
        "title": "Real-Time Fraud Detection with ML",
        "description": "Detect fraudulent transactions in real-time using ML models. "
                       "Process 10K transactions per second with <100ms latency.",
        "tech_stack": "XGBoost, Kafka, Redis, Python",
        "timeline": "6 months",
        "team_size": 4,
        "budget": "$200K",
        "data_available": "2 years of transaction data, 0.1% fraud rate, PII restrictions",
    },
    {
        "title": "Internal Code Review AI",
        "description": "AI that reviews pull requests and suggests improvements. "
                       "Integrate with GitHub, learn from team's coding standards.",
        "tech_stack": "GPT-4, GitHub API, Python",
        "timeline": "2 months",
        "team_size": 1,
        "budget": "$5K",
        "data_available": "GitHub repo history, style guide document",
    },
]
print(f"Proposals to evaluate: {len(proposals)}")

## Step 2 — Feasibility Analysis

In [ ]:
class FeasibilityDimension(BaseModel):
    dimension: str = Field(description="technical, data, resource, timeline, business")
    score: float = Field(ge=0, le=1)
    assessment: str
    blockers: list[str]

class ProjectCritique(BaseModel):
    title: str
    overall_feasibility: float = Field(ge=0, le=1)
    dimensions: list[FeasibilityDimension]
    strengths: list[str]
    weaknesses: list[str]
    risks: list[str]
    missing_requirements: list[str]
    recommendation: str = Field(description="proceed, proceed_with_changes, delay, reject")
    suggested_changes: list[str]

critic = llm.with_structured_output(ProjectCritique)

critiques = []
for proposal in proposals:
    critique = critic.invoke(
        f"Critically evaluate this AI project proposal:\n\n"
        f"{json.dumps(proposal, indent=2)}"
    )
    critiques.append(critique)
    print(f"\n{'='*50}")
    print(f"{critique.title}")
    print(f"  Feasibility: {critique.overall_feasibility:.0%}")
    print(f"  Recommendation: {critique.recommendation.upper()}")
    for dim in critique.dimensions:
        bar = "█" * int(dim.score * 10)
        print(f"    {dim.dimension:<12} {bar} {dim.score:.0%}")

## Step 3 — Risk Mitigation Plans

In [ ]:
risk_prompt = ChatPromptTemplate.from_messages([
    ("system", "For each risk, create a specific mitigation plan with "
     "actions, owners, and success criteria."),
    ("human", "Project: {title}\nRisks: {risks}\n\nMitigation plans:")
])
risk_chain = risk_prompt | llm | StrOutputParser()

for critique in critiques:
    if critique.risks:
        plan = risk_chain.invoke({
            "title": critique.title,
            "risks": json.dumps(critique.risks),
        })
        print(f"\n{'='*50}")
        print(f"RISK MITIGATION: {critique.title}")
        print(plan[:400])

## Step 4 — Comparison Dashboard

In [ ]:
rows = []
for c in critiques:
    row = {"project": c.title[:30], "feasibility": f"{c.overall_feasibility:.0%}",
           "recommendation": c.recommendation, "risks": len(c.risks),
           "strengths": len(c.strengths), "weaknesses": len(c.weaknesses)}
    for dim in c.dimensions:
        row[dim.dimension] = f"{dim.score:.0%}"
    rows.append(row)

df = pd.DataFrame(rows)
print("PROJECT COMPARISON DASHBOARD")
print("=" * 60)
print(df.to_string(index=False))

# Ranking
print(f"\nRANKING (by feasibility):")
for i, c in enumerate(sorted(critiques, key=lambda x: x.overall_feasibility, reverse=True), 1):
    print(f"  {i}. {c.title} — {c.overall_feasibility:.0%} [{c.recommendation}]")
    if c.suggested_changes:
        print(f"     Changes: {c.suggested_changes[0]}")

## What You Learned
- **Multi-dimensional feasibility analysis** for AI projects
- **Risk identification and mitigation planning**
- **Project comparison and ranking**
- **Structured recommendation framework** (proceed/delay/reject)